# 🔭 AI Anomaly Hunter — Kaggle Cloud Training
**Conv1D Autoencoder | Kepler / TESS Lichtkurven | GPU T4**

Dieses Notebook klont das Repo, installiert alle Abhängigkeiten und führt die komplette Pipeline aus:
1. **HARVEST** — Lichtkurven von NASA via `lightkurve` herunterladen
2. **TRAIN** — Autoencoder auf GPU trainieren
3. **HUNT** — Anomalien in 5000 Sternen suchen

> ⚡ Stelle sicher, dass du unter *Settings → Accelerator* **GPU T4 x2** aktiviert hast!

In [ ]:
# ── Zelle 1: GPU prüfen ──────────────────────────────────────────────
import subprocess, sys
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️  Kein GPU gefunden — bitte Accelerator in den Notebook-Settings aktivieren!')

In [ ]:
# ── Zelle 2: Abhängigkeiten installieren ─────────────────────────────
# lightkurve und astropy sind auf Kaggle nicht vorinstalliert
import sys
!{sys.executable} -m pip install -q lightkurve astropy
print('✅ Installation abgeschlossen')

In [ ]:
# ── Zelle 3: Repo klonen ──────────────────────────────────────────────
import os
REPO = 'https://github.com/Youssef-Bou/AI-anomaly-hunter.git'
WORKDIR = '/kaggle/working/AI-anomaly-hunter'

if not os.path.exists(WORKDIR):
    !git clone {REPO} {WORKDIR}
else:
    !git -C {WORKDIR} pull

os.chdir(WORKDIR)
print(f'📁 Arbeitsverzeichnis: {os.getcwd()}')
!ls -lh

In [ ]:
# ── Zelle 4: TensorFlow GPU-Check ────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow {tf.__version__}')
print(f'GPUs erkannt: {gpus}')
if gpus:
    print('✅ Training wird auf GPU ausgeführt')
else:
    print('⚠️  Kein GPU — Training läuft auf CPU (langsamer)')

In [ ]:
# ── Zelle 5: train_candidates.csv erstellen (Kepler-Sterne ohne Transit-Flag) ──
# Wir ziehen bekannte 'normale' Sterne direkt aus dem NASA Exoplanet Archive
import pandas as pd, requests, io

N_TRAIN = 2000   # <-- erhöhe auf 5000 für besseres Modell (braucht ~2h)
N_HUNT  = 3000   # <-- Sterne für die Anomalie-Suche

print('⬇️  Lade Kepler-Sterntabelle vom NASA Exoplanet Archive...')
url = (
    'https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query='
    'select+kepid+from+keplerstellar+where+logg+is+not+null'
    f'+order+by+kepid&format=csv&top={N_TRAIN + N_HUNT}'
)
resp = requests.get(url, timeout=60)
resp.raise_for_status()
df_all = pd.read_csv(io.StringIO(resp.text), comment='#')
df_all.columns = df_all.columns.str.strip()
print(f'Gesamt: {len(df_all)} Sterne geladen')

# Aufteilen: erste N_TRAIN zum Trainieren, Rest zum Jagen
df_train = df_all.iloc[:N_TRAIN][['kepid']]
df_hunt  = df_all.iloc[N_TRAIN:N_TRAIN + N_HUNT][['kepid']]

df_train.to_csv('train_candidates.csv', index=False)
df_hunt.to_csv('search_targets.csv', index=False)
print(f'✅ train_candidates.csv: {len(df_train)} Zeilen')
print(f'✅ search_targets.csv:   {len(df_hunt)} Zeilen')

In [ ]:
# ── Zelle 6: Phase 1 — HARVEST (Lichtkurven downloaden) ──────────────
# Laufzeit: ~30–60 min für 2000 Sterne (NASA-API limitiert)
!python TRAIN_HUNT_pipeline.py --phase harvest --n-train-download {N_TRAIN} --log-level INFO

In [ ]:
# ── Zelle 7: Phase 2 — TRAIN (Modell auf GPU trainieren) ─────────────
# Laufzeit: ~15–30 min auf T4 GPU für 2000 Sterne, 100 Epochs
!python TRAIN_HUNT_pipeline.py --phase train --epochs 100 --batch-size 64 --log-level INFO

In [ ]:
# ── Zelle 8: Trainings-Ergebnis prüfen ───────────────────────────────
import os, numpy as np

model_size = os.path.getsize('TRAIN_model.keras') / 1024
print(f'✅ TRAIN_model.keras gespeichert ({model_size:.1f} KB)')

if os.path.exists('mse_threshold.npy'):
    t = float(np.load('mse_threshold.npy'))
    print(f'✅ Auto-Threshold (95th pct val MSE): {t:.6f}')
else:
    print('⚠️  mse_threshold.npy nicht gefunden')

In [ ]:
# ── Zelle 9: Phase 3 — HUNT (Anomalien suchen) ───────────────────────
# Laufzeit: ~60–90 min für 3000 Sterne
!python TRAIN_HUNT_pipeline.py --phase hunt --n-search-analysis {N_HUNT} --log-level INFO

In [ ]:
# ── Zelle 10: Ergebnisse anzeigen & als Kaggle Output speichern ───────
import pandas as pd, shutil, os

df = pd.read_csv('anomalie_ergebnisse.csv')
print(f'Insgesamt analysiert: {len(df)} Sterne')
print(f'Anomalien gefunden:   {df["Anomaly"].sum() if "Anomaly" in df.columns else "N/A"}')
print()
print('🏆 Top 20 Anomalie-Kandidaten:')
display(df.head(20))

# Output-Dateien in /kaggle/working kopieren (werden als Download-Artefakte gespeichert)
for fname in ['anomalie_ergebnisse.csv', 'TRAIN_model.keras', 'mse_threshold.npy']:
    src = fname
    dst = f'/kaggle/working/{fname}'
    if os.path.exists(src) and os.path.abspath(src) != os.path.abspath(dst):
        shutil.copy(src, dst)
        print(f'📦 {fname} → /kaggle/working/ (downloadbar)')
    else:
        print(f'✅ {fname} bereits in /kaggle/working/')

In [ ]:
# ── Zelle 11: Schnellcheck — Beispiel-Stern analysieren ──────────────
# Teste das gespeicherte Modell direkt hier im Notebook
import numpy as np, lightkurve as lk
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt

TEST_STAR = 'KIC 8462852'   # Tabbys Star — bekannte Anomalie!
DATA_POINTS = 1000

model = load_model('TRAIN_model.keras')

search = lk.search_lightcurve(TEST_STAR, author='Kepler', quarter=8)
lc = search.download().remove_nans().normalize().flatten(window_length=401)
y = np.interp(np.linspace(0,1,DATA_POINTS), np.linspace(0,1,len(lc.flux)), lc.flux.value).astype(np.float32)

y_pred = model.predict(y.reshape(1, DATA_POINTS, 1), verbose=0).flatten()
mse = float(np.mean(np.square(y - y_pred)))
max_dip = float(1.0 - np.min(y))

if os.path.exists('mse_threshold.npy'):
    thr = float(np.load('mse_threshold.npy'))
    verdict = '⚠️ ANOMALIE!' if mse > thr else '✅ Normal'
    print(f'Threshold: {thr:.6f}  |  MSE: {mse:.6f}  →  {verdict}')
else:
    print(f'MSE: {mse:.6f}  |  Max Dip: {max_dip:.2%}')

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
axes[0].plot(y, lw=0.8, color='steelblue', label='NASA-Daten')
axes[0].plot(y_pred, lw=1.2, color='orange', linestyle='--', label='KI-Rekonstruktion')
axes[0].set_title(f'{TEST_STAR} — Lichtkurve vs. Rekonstruktion')
axes[0].legend(); axes[0].grid(alpha=0.3)
residual = np.abs(y - y_pred)
axes[1].bar(range(len(residual)), residual, color='tomato', width=1)
axes[1].set_title('Residuum |Real − Rekonstruktion|')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/quickcheck_plot.png', dpi=150)
plt.show()
print('📊 Plot gespeichert: /kaggle/working/quickcheck_plot.png')